# BWF Player Lookup

Type a badminton player's name and get their **personal details** (nationality, height, playing hand) and **ranking** (current rank and how long they have held it) from bwfbadminton.com. Section 4 downloads the player's **tournament history for the last year**: the result in each tournament, who they played with, who they played against, the score of every game, and the **game details** of every match (the Match tab and each Game tab of the site's match page, including the score after every rally). It is saved to a database and CSV files. Section 5 shows the details of one match, and section 6 puts two players side by side (Jonatan Christie and An Se Young) using the saved data.

**How to use:** set `PLAYER_NAME` in section 1, then *Run All*. Every value the site does not list is shown as `null`, with a note. Section 4 makes about 85 requests for a busy singles player (roughly four minutes at the polite pace; repeating it is quick because responses are cached). Set `GAME_DETAILS = False` there to skip the game details (about 25 requests).

The notebook is a thin interface: all logic lives in the `bwf_player` package (see the README).

In [1]:
import logging
import sqlite3

from bwf_player import (
    BlockedByCloudflareError,
    BwfClientError,
    download_player_history,
    format_history,
    format_match_details,
    format_result,
    get_match_details,
    lookup_player,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Enter a player name

In [2]:
# Any reasonable spelling works: case, accents, reversed order, small typos ("jonathan cristie").
PLAYER_NAME = "jonathan cristie"

# None = the first ranking event the site lists (usually singles). To pick another event, copy the
# id shown as "Other event" in the result, e.g. "9-90070" (a doubles event).
EVENT_ID = None

## 2. Result

In [3]:
try:
    result = lookup_player(PLAYER_NAME, event_id=EVENT_ID)
except BlockedByCloudflareError as exc:
    result = None
    print(f"Blocked by Cloudflare. Wait a while (or switch network) before trying again.\n{exc}")
except BwfClientError as exc:
    result = None
    print(f"The request failed: {exc}")
else:
    print(format_result(result))

Search:         FOUND - Matched 'Jonatan CHRISTIE' (score 94).
Profile URL:    https://bwfbadminton.com/player/73442/jonatan-christie

Personal details
  Name:          Jonatan CHRISTIE
  Nationality:   Indonesia
  Height:        179.0 cm
  Playing hand:  Right

Ranking (MEN'S SINGLES)
  Current rank:  1
  At this rank:  5 week(s), since 2026-08-25 (latest ranking list 2026-09-22)


## 3. The same result as structured data (JSON)

In [4]:
print(result.model_dump_json(indent=2) if result else "No result.")

{
  "search": {
    "query": "jonathan cristie",
    "status": "found",
    "best_match": {
      "player_id": "73442",
      "slug": "jonatan-christie",
      "name": "Jonatan CHRISTIE",
      "country": null,
      "profile_url": "https://bwfbadminton.com/player/73442/jonatan-christie",
      "score": 93.8
    },
    "candidates": [
      {
        "player_id": "73442",
        "slug": "jonatan-christie",
        "name": "Jonatan CHRISTIE",
        "country": null,
        "profile_url": "https://bwfbadminton.com/player/73442/jonatan-christie",
        "score": 93.8
      }
    ],
    "message": "Matched 'Jonatan CHRISTIE' (score 94)."
  },
  "profile": {
    "player_id": "73442",
    "player_found": true,
    "name": "Jonatan CHRISTIE",
    "nationality": "Indonesia",
    "height_cm": 179.0,
    "playing_hand": "Right",
    "missing_fields": [],
    "notes": []
  },
  "ranking": {
    "player_id": "73442",
    "event": {
      "id": "6-0",
      "name": "MEN'S SINGLES"
    },
    "o

## 4. Tournament history for the last year

Downloads every tournament the player entered in the window and, for each event, all matches: the **result** (`1st`, `QF`, `R16`, ...), the **partner** (doubles), the **opponents** and the **points of every game**. Unless you switch it off, it also downloads the **game details** of each played match: what the site's match page shows (Match tab, Game 1, Game 2, ...), including the score after every rally. Everything is saved to a SQLite database and exported as CSV.

- `HISTORY_PLAYER`: a name, or the site's player id as a number.
- `SINCE` / `UNTIL`: `None` means the last 12 months up to today. A tournament counts if its dates overlap the window.
- `GAME_DETAILS`: `True` also downloads the game details (one more request per played match).
- Running it again never duplicates anything; it updates the same rows.

In [5]:
HISTORY_PLAYER = PLAYER_NAME      # or the site's player id, e.g. 73442

# None = one year back from today, up to today. Or pass datetime.date values.
SINCE = UNTIL = None

# True = also download the game details of every played match (Match tab, Game tabs, every rally).
GAME_DETAILS = True

# Where to save (the data/ folder is git-ignored).
DB_PATH = "data/bwf_notebook.sqlite"   # the notebook keeps its own file; delete it to start from scratch
CSV_DIR = "data/notebook_export"

In [6]:
try:
    history = download_player_history(
        HISTORY_PLAYER, since=SINCE, until=UNTIL, db_path=DB_PATH, export_dir=CSV_DIR, game_details=GAME_DETAILS
    )
except BlockedByCloudflareError as exc:
    history = None
    print(f"Blocked by Cloudflare. Wait a while (or switch network) before trying again.\n{exc}")
except BwfClientError as exc:
    history = None
    print(f"The download failed: {exc}")
else:
    print(format_history(history))

INFO bwf_player.history: Looking up the tournaments of player 73442


INFO bwf_player.history: [1/19] 2025-09-23 SUWON VICTOR Korea Open 2025 (MS)


INFO bwf_player.history:     match 1/5: R32 (code 59), game details


INFO bwf_player.history:     match 2/5: R16 (code 43), game details


INFO bwf_player.history:     match 3/5: QF (code 35), game details


...

INFO bwf_player.history:     match 2/2: R16 (code 8), game details


INFO bwf_player.history: [19/19] 2026-09-20 20th Asian Games Aichi-Nagoya 2026 (Team) (Singles)


INFO bwf_player.history:     match 1/2: QF (code 41), game details


INFO bwf_player.history:     match 2/2: SF (code 61), game details


Search:       FOUND - Matched 'Jonatan CHRISTIE' (score 94).
Player:       Jonatan CHRISTIE (id 73442)
Window:       2025-09-24 to 2026-09-24
Downloaded:   19 tournament(s), 19 event(s), 58 match(es) (58 played), 139 game(s)
Checked:      the matches reproduce the site's own totals: yes (19 event(s))
Game details: 58 match(es), 56 with rally data, 2 with game scores only, 4779 rallies
Checked:      rallies, statistics and scores agree with each other and with the player's page: yes
Database:     data\bwf_notebook.sqlite
CSV:          data\notebook_export\results.csv
CSV:          data\notebook_export\matches.csv
CSV:          data\notebook_export\games.csv
CSV:          data\notebook_export\match_stats.csv
CSV:          data\notebook_export\game_stats.csv
CSV:          data\notebook_export\rallies.csv

2025-09-23  SUWON VICTOR Korea Open 2025  [MS]  result: 1st  5-0 in matches  (HSBC BWF World Tour Super 500)
    R32       won              vs NG Ka Long Angus  21-11, 21-17
    R16     

### Read the saved data back

The database has the tables `players`, `tournaments`, `results`, `matches`, `match_players` and `games`, the game-detail tables `match_stats`, `game_stats` and `rallies`, and views with one row per player and match (`player_match_view`, `player_match_stats_view`), per game (`player_game_view`) and per rally (`player_rally_view`). For analysis, load the CSV files or the views, for example `pandas.read_sql("SELECT * FROM player_match_view", sqlite3.connect(DB_PATH))`.

In [7]:
if history and history.database:
    with sqlite3.connect(history.database) as connection:
        rows = connection.execute(
            """SELECT match_date, tournament, event_code, round, won, partner, opponent_1, opponent_2, games
               FROM player_match_view WHERE player_id = ? ORDER BY match_date, seq LIMIT 12""",
            (int(history.player_id),),
        ).fetchall()
    print("First matches in the database (date, tournament, event, round, won, partner, opponents, games):")
    for row in rows:
        print("  ", " | ".join("-" if value is None else str(value) for value in row))
    print("\nCSV files:", *history.csv_files.values(), sep="\n  ")
else:
    print("Nothing was saved (see the messages above).")

First matches in the database (date, tournament, event, round, won, partner, opponents, games):
   2025-09-24 | SUWON VICTOR Korea Open 2025 | MS | R32 | 1 | - | NG Ka Long Angus | - | 21-11, 21-17
   2025-09-25 | SUWON VICTOR Korea Open 2025 | MS | R16 | 1 | - | Chia Hao LEE | - | 22-20, 15-21, 21-15
   2025-09-26 | SUWON VICTOR Korea Open 2025 | MS | QF | 1 | - | Kenta NISHIMOTO | - | 21-14, 21-8
   2025-09-27 | SUWON VICTOR Korea Open 2025 | MS | SF | 1 | - | Alwi FARHAN | - | 18-21, 21-14, 21-15
   2025-09-28 | SUWON VICTOR Korea Open 2025 | MS | Final | 1 | - | Anders ANTONSEN | - | 21-10, 15-21, 21-17
   2025-10-15 | VICTOR Denmark Open 2025 | MS | R32 | 1 | - | Kenta NISHIMOTO | - | 10-21, 21-11, 21-7
   2025-10-16 | VICTOR Denmark Open 2025 | MS | R16 | 1 | - | Kodai NARAOKA | - | 21-7, 21-13
   2025-10-17 | VICTOR Denmark Open 2025 | MS | QF | 1 | - | LI Shi Feng | - | 21-18, 21-23, 21-17
   2025-10-18 | VICTOR Denmark Open 2025 | MS | SF | 1 | - | Alex LANIER | - | 11-21, 21-

## 5. The game details of one match

The Match tab and each Game tab of a match page, for example [All England 2026, round of 16](https://bwfworldtour.bwfbadminton.com/tournament/5515/all-england-open-badminton-championships-2026/match/13). The two numbers are in the page address: the tournament (`5515`) and the match (`match/13`). Side 1 and side 2 are the site's own order (team 1, team 2). Below the tabs, the score after every rally is listed (the chart on the page).

The figures are **checked**: the statistics are re-derived from the rally sequence and compared with what the site shows, and with the match on the player's page. `Checks: ... agree` at the end means everything matched.

In [8]:
TOURNAMENT_ID = 5515      # the number after /tournament/ in the page address
MATCH_CODE = 13           # the number after /match/

try:
    details = get_match_details(TOURNAMENT_ID, MATCH_CODE)
except BlockedByCloudflareError as exc:
    details = None
    print(f"Blocked by Cloudflare. Wait a while (or switch network) before trying again.\n{exc}")
except BwfClientError as exc:
    details = None
    print(f"Could not get the match details: {exc}")
else:
    print(format_match_details(details))

All England Open Badminton Championships 2026 | MS | R16 | 2026-03-05 19:15 | Utilita Arena Birmingham | 48 min
  side 1: LIN Chun-Yi
  side 2: Jonatan CHRISTIE
  winner: LIN Chun-Yi

MATCH
                              side 1  side 2
  Final match score                2       0
  Game 1 score                    21      19
  Game 2 score                    21      12
  Game points                      7       0
  Most consecutive points          7       5
  Total points played             73      73
  Total points won                42      31

GAME 1
                              side 1  side 2
  Score                           21      19
  Most consecutive points          7       5
  Game points                      6       0
  Total points played             40      40
  Total points won                21      19
  score after each rally: 0-1 1-1 1-2 2-2 3-2 3-3 3-4 4-4 5-4 5-5 6-5 6-6 7-6 8-6 9-6 9-7 9-8 9-9 9-10 9-11 10-11 11-11 12-11 13-11 14-11 15-11 16-11 16-12 16-13 17-13 17-1

### The same match in the database

If the match belongs to the player downloaded in section 4, its games are in the view `player_game_view`, from the player's point of view (the player's figures first).

In [9]:
if history and history.database:
    with sqlite3.connect(history.database) as connection:
        games = connection.execute(
            """SELECT game_no, player_points, opponent_points, total_points_played, player_consecutive_points,
                      opponent_consecutive_points, player_game_points, opponent_game_points
               FROM player_game_view WHERE tournament_id = ? AND match_code = ? ORDER BY game_no""",
            (TOURNAMENT_ID, str(MATCH_CODE)),
        ).fetchall()
        rallies = connection.execute(
            "SELECT COUNT(*) FROM player_rally_view WHERE tournament_id = ? AND match_code = ?", (TOURNAMENT_ID, str(MATCH_CODE))
        ).fetchone()[0]
    if games:
        print("game | points | opponent | rallies | longest run (player / opponent) | game points (player / opponent)")
        for game_no, mine, theirs, total, run_mine, run_theirs, gp_mine, gp_theirs in games:
            print(f"  {game_no}  |  {mine:>2}    |   {theirs:>2}     |   {total}    |   {run_mine} / {run_theirs}   |   {gp_mine} / {gp_theirs}")
        print(f"{rallies} rallies are stored for this match.")
    else:
        print("This match is not in the database (it belongs to another player, or the game details were switched off).")
else:
    print("Nothing was saved (see the messages above).")

game | points | opponent | rallies | longest run (player / opponent) | game points (player / opponent)
  1  |  19    |   21     |   40    |   5 / 7   |   0 / 6
  2  |  12    |   21     |   33    |   4 / 5   |   0 / 1
73 rallies are stored for this match.


## 6. Two players side by side

The saved data answers questions the site itself does not. Here the history and game details of a second player, **An Se Young**, are downloaded into the same database as the player from section 4, and the two are compared with plain SQL. Change `SECOND_PLAYER` to any name or player id. (About 90 requests, a few minutes the first time; cached afterwards.)

All the tables below come from the views `player_match_view`, `player_match_stats_view`, `player_game_view` and `player_rally_view`, which show every figure from the player's own point of view.

In [10]:
SECOND_PLAYER = "An Se Young"

try:
    second = download_player_history(SECOND_PLAYER, since=SINCE, until=UNTIL, db_path=DB_PATH, export_dir=CSV_DIR, game_details=GAME_DETAILS)
except BlockedByCloudflareError as exc:
    second = None
    print(f"Blocked by Cloudflare. Wait a while (or switch network) before trying again.\n{exc}")
except BwfClientError as exc:
    second = None
    print(f"The download failed: {exc}")
else:
    print(format_history(second, matches=False))

INFO bwf_player.history: Looking up the tournaments of player 87442


INFO bwf_player.history: [1/17] 2025-09-23 SUWON VICTOR Korea Open 2025 (WS)


INFO bwf_player.history:     match 1/5: R32 (code 119), game details


INFO bwf_player.history:     match 2/5: R16 (code 111), game details


INFO bwf_player.history:     match 3/5: QF (code 107), game details


...

INFO bwf_player.history:     match 5/5: Final (code 64), game details


INFO bwf_player.history: [17/17] 2026-09-20 20th Asian Games Aichi-Nagoya 2026 (Team) (Singles)


INFO bwf_player.history:     match 1/2: QF (code 126), game details


INFO bwf_player.history:     match 2/2: SF (code 141), game details


Search:       FOUND - Matched 'AN Se Young' (score 100).
Player:       AN Se Young (id 87442)
Window:       2025-09-24 to 2026-09-24
Downloaded:   17 tournament(s), 17 event(s), 79 match(es) (76 played, 1 retired, 2 walkover), 168 game(s)
Checked:      the matches reproduce the site's own totals: yes (17 event(s))
Game details: 77 match(es), 75 with rally data, 2 with game scores only, 5439 rallies, 2 not requested (byes, walkovers)
Checked:      rallies, statistics and scores agree with each other and with the player's page: yes
Database:     data\bwf_notebook.sqlite
CSV:          data\notebook_export\results.csv
CSV:          data\notebook_export\matches.csv
CSV:          data\notebook_export\games.csv
CSV:          data\notebook_export\match_stats.csv
CSV:          data\notebook_export\game_stats.csv
CSV:          data\notebook_export\rallies.csv

2025-09-23  SUWON VICTOR Korea Open 2025  [WS]  result: 2nd  4-1 in matches  (HSBC BWF World Tour Super 500)

2025-10-14  VICTOR Denmark 

In [11]:
def show(title, sql, params=()):
    """Run a query on the saved data and print it as an aligned table."""
    with sqlite3.connect(DB_PATH) as connection:
        cursor = connection.execute(sql, params)
        columns = [column[0] for column in cursor.description]
        table = [["-" if value is None else str(value) for value in row] for row in cursor.fetchall()]
    widths = [max(len(name), *(len(row[i]) for row in table)) if table else len(name) for i, name in enumerate(columns)]
    print(f"\n{title}")
    print("  ".join(name.ljust(width) for name, width in zip(columns, widths)))
    for row in table:
        print("  ".join(value.ljust(width) for value, width in zip(row, widths)))


PLAYER_IDS = (int(history.player_id), int(second.player_id)) if history and second else ()

### Record and results

In [12]:
if PLAYER_IDS:
    show("Matches played in the window",
         """SELECT player_name AS player, COUNT(*) AS matches, SUM(won) AS won, COUNT(*) - SUM(won) AS lost,
                   ROUND(100.0 * SUM(won) / COUNT(*), 1) AS win_pct
            FROM player_match_view WHERE player_id IN (?, ?) AND status IN ('played', 'retired') GROUP BY player_id""",
         PLAYER_IDS)
    show("Results in their events",
         """SELECT p.name AS player, COUNT(*) AS events, SUM(r.position = '1st') AS titles, SUM(r.position = '2nd') AS runner_up,
                   SUM(r.position = '3rd') AS third
            FROM results r JOIN players p ON p.player_id = r.player_id WHERE r.player_id IN (?, ?) GROUP BY r.player_id""",
         PLAYER_IDS)
    show("The titles",
         """SELECT p.name AS player, t.start_date AS date, substr(t.name, 1, 44) AS tournament, r.event_code AS event,
                   r.matches_won || '-' || r.matches_lost AS record
            FROM results r JOIN tournaments t ON t.tournament_id = r.tournament_id JOIN players p ON p.player_id = r.player_id
            WHERE r.player_id IN (?, ?) AND r.position = '1st' ORDER BY r.player_id, t.start_date""",
         PLAYER_IDS)


Matches played in the window
player            matches  won  lost  win_pct
Jonatan CHRISTIE  58       38   20    65.5   
AN Se Young       77       75   2     97.4   

Results in their events
player            events  titles  runner_up  third
Jonatan CHRISTIE  19      3       2          1    
AN Se Young       17      11      2          0    

The titles
player            date        tournament                                    event  record
Jonatan CHRISTIE  2025-09-23  SUWON VICTOR Korea Open 2025                  MS     5-0   
Jonatan CHRISTIE  2025-10-14  VICTOR Denmark Open 2025                      MS     5-0   
Jonatan CHRISTIE  2025-10-28  HYLO Open 2025                                MS     5-0   
AN Se Young       2025-10-14  VICTOR Denmark Open 2025                      WS     5-0   
AN Se Young       2025-10-21  YONEX French Open 2025                        WS     5-0   
AN Se Young       2025-11-18  SATHIO GROUP Australian Open 2025             WS     5-0   
AN Se Young 

### What the game details add

Only matches where the site tracks rallies are counted here (the others have empty statistics). `points_won_pct` is the share of all rallies the player won; `best_run` is the most consecutive points in one match.

In [13]:
if PLAYER_IDS:
    show("Rally statistics per player (matches with rally data)",
         """SELECT v.player_name AS player, COUNT(*) AS matches, ROUND(AVG(v.duration_min), 1) AS avg_minutes,
                   ROUND(AVG(s.player_rallies_played), 1) AS avg_rallies,
                   ROUND(100.0 * SUM(s.player_rallies_won) / SUM(s.player_rallies_played), 1) AS points_won_pct,
                   MAX(s.player_consecutive_points) AS best_run, MAX(s.opponent_consecutive_points) AS worst_run_against
            FROM player_match_stats_view s JOIN player_match_view v ON v.match_id = s.match_id AND v.player_id = s.player_id
            WHERE s.tracked = 1 AND s.player_id IN (?, ?) GROUP BY s.player_id""",
         PLAYER_IDS)
    show("The longest games (most rallies)",
         """SELECT v.player_name AS player, g.match_date AS date, substr(g.tournament, 1, 26) AS tournament, g.round,
                   v.opponent_1 AS opponent, g.game_no AS game, g.player_points || '-' || g.opponent_points AS score,
                   g.total_points_played AS rallies
            FROM player_game_view g JOIN player_match_view v ON v.match_id = g.match_id AND v.player_id = g.player_id
            WHERE g.player_id IN (?, ?) AND g.total_points_played IS NOT NULL ORDER BY g.total_points_played DESC LIMIT 6""",
         PLAYER_IDS)
    show("Comeback games: won after trailing by 8 points or more (needs the rally-by-rally data)",
         """SELECT v.player_name AS player, g.match_date AS date, substr(g.tournament, 1, 26) AS tournament, g.round,
                   v.opponent_1 AS opponent, g.game_no AS game, g.player_points || '-' || g.opponent_points AS score,
                   MAX(r.opponent_points - r.player_points) AS max_deficit
            FROM player_game_view g
            JOIN player_match_view v ON v.match_id = g.match_id AND v.player_id = g.player_id
            JOIN player_rally_view r ON r.match_id = g.match_id AND r.player_id = g.player_id AND r.game_no = g.game_no
            WHERE g.player_id IN (?, ?) AND g.player_points > g.opponent_points
            GROUP BY g.match_id, g.player_id, g.game_no HAVING MAX(r.opponent_points - r.player_points) >= 8
            ORDER BY max_deficit DESC LIMIT 6""",
         PLAYER_IDS)


Rally statistics per player (matches with rally data)
player            matches  avg_minutes  avg_rallies  points_won_pct  best_run  worst_run_against
Jonatan CHRISTIE  56       53.9         85.3         52.8            10        9                
AN Se Young       75       45.4         72.5         61.6            16        10               

The longest games (most rallies)
player            date        tournament                  round  opponent       game  score  rallies
Jonatan CHRISTIE  2025-10-23  YONEX French Open 2025      R16    Koki WATANABE  3     23-25  48     
Jonatan CHRISTIE  2026-06-05  POLYTRON Indonesia Open 20  QF     Yushi TANAKA   2     24-22  46     
AN Se Young       2025-10-19  VICTOR Denmark Open 2025    Final  WANG Zhi Yi    2     24-22  46     
AN Se Young       2026-01-11  PETRONAS Malaysia Open 202  Final  WANG Zhi Yi    2     24-22  46     
AN Se Young       2026-08-22  BWF World Championships 20  SF     WANG Zhi Yi    1     24-22  46     
Jonatan CHRIST

### Sample matches with their Match tab figures

The latest five played matches of each player: the games, the length, the rallies played and the longest run of points for the player and for the opponent.

In [14]:
for player_id in PLAYER_IDS:
    show(f"Latest matches of player {player_id}",
         """SELECT v.match_date AS date, substr(v.tournament, 1, 28) AS tournament, v.round, v.won, v.opponent_1 AS opponent,
                   v.games, v.duration_min AS min, s.player_rallies_played AS rallies,
                   s.player_consecutive_points AS run, s.opponent_consecutive_points AS run_against
            FROM player_match_view v LEFT JOIN player_match_stats_view s ON s.match_id = v.match_id AND s.player_id = v.player_id
            WHERE v.player_id = ? AND v.status = 'played' ORDER BY v.match_date DESC, v.seq DESC LIMIT 5""",
         (player_id,))


Latest matches of player 73442
date        tournament                    round  won  opponent            games                min  rallies  run  run_against
2026-09-23  20th Asian Games Aichi-Nagoy  SF     0    Kunlavut VITIDSARN  17-21, 14-21         53   -        -    -          
2026-09-22  20th Asian Games Aichi-Nagoy  QF     0    LEONG Jun Hao       22-20, 12-21, 17-21  84   -        -    -          
2026-09-03  LI-NING China Masters 2026    R16    0    Jason GUNAWAN       16-21, 16-21         45   74       4    6          
2026-09-01  LI-NING China Masters 2026    R32    1    LEONG Jun Hao       21-17, 21-19         44   78       7    4          
2026-08-20  BWF World Championships 2026  R16    0    Rasmus GEMKE        21-11, 13-21, 18-21  70   105      9    5          

Latest matches of player 87442
date        tournament                    round  won  opponent                games               min  rallies  run  run_against
2026-09-23  20th Asian Games Aichi-Nagoy  SF     1 

### Sample rally data

The first rallies of one game, from the player's point of view (`won` = 1 when the player won the rally). Here the first game of the match from section 5.

In [15]:
if PLAYER_IDS:
    show(f"Rallies of game 1, tournament {TOURNAMENT_ID}, match {MATCH_CODE} (first 12 of the game)",
         """SELECT v.player_name AS player, r.rally_no, r.player_points AS player_pts, r.opponent_points AS opponent_pts,
                   r.rally_won_by_player AS won
            FROM player_rally_view r JOIN player_match_view v ON v.match_id = r.match_id AND v.player_id = r.player_id
            WHERE r.tournament_id = ? AND r.match_code = ? AND r.game_no = 1 AND r.player_id IN (?, ?)
            ORDER BY r.rally_no LIMIT 12""",
         (TOURNAMENT_ID, str(MATCH_CODE), *PLAYER_IDS))


Rallies of game 1, tournament 5515, match 13 (first 12 of the game)
player            rally_no  player_pts  opponent_pts  won
Jonatan CHRISTIE  1         1           0             1  
Jonatan CHRISTIE  2         1           1             0  
Jonatan CHRISTIE  3         2           1             1  
Jonatan CHRISTIE  4         2           2             0  
Jonatan CHRISTIE  5         2           3             0  
Jonatan CHRISTIE  6         3           3             1  
Jonatan CHRISTIE  7         4           3             1  
Jonatan CHRISTIE  8         4           4             0  
Jonatan CHRISTIE  9         4           5             0  
Jonatan CHRISTIE  10        5           5             1  
Jonatan CHRISTIE  11        5           6             0  
Jonatan CHRISTIE  12        6           6             1  


## More examples

The same lookup for other cases: a left-handed player, a retired (unranked) player, a player whose profile lists nothing, a name that matches several players, and a name that matches nobody.

In [16]:
EXAMPLES = ["Carolina Marin", "Chong Wei Lee", "Aadhya Shine", "christie", "not a real player"]

for name in EXAMPLES:
    print("=" * 72)
    print(f"Query: {name!r}")
    try:
        print(format_result(lookup_player(name)))
    except BwfClientError as exc:
        print(f"The request failed: {exc}")

Query: 'Carolina Marin'
Search:         FOUND - Matched 'Carolina MARIN' (score 100).
Profile URL:    https://bwfbadminton.com/player/18228/carolina-marin

Personal details
  Name:          Carolina MARIN
  Nationality:   Spain
  Height:        172.0 cm
  Playing hand:  Left

Ranking (WOMEN'S SINGLES)
  Current rank:  null
  At this rank:  null
  Other event:   WOMEN'S DOUBLES (Beatriz CORRALES) [9-95780]
  Other event:   WOMEN'S DOUBLES (Sara PEÑALVER) [9-76747]
  Other event:   WOMEN'S DOUBLES (Isabel FERNANDEZ) [9-90558]
  Other event:   WOMEN'S DOUBLES (Clara AZURMENDI) [9-74218]
  Other event:   WOMEN'S DOUBLES (Ana Maria MARTIN) [9-55582]

Notes
  - Not currently ranked in WOMEN'S SINGLES: the site lists no current rank.
Query: 'Chong Wei Lee'
Search:         FOUND - Matched 'LEE Chong Wei' (score 100).
Profile URL:    https://bwfbadminton.com/player/50152/lee-chong-wei

Personal details
  Name:          LEE Chong Wei
  Nationality:   Malaysia
  Height:        172.0 cm
  Playing 

## Notes

- **Search** ignores case, accents and word order, and tolerates typos in full names. A partial name such as "christie" returns ranked candidates instead of guessing.
- **Weeks at this rank** is derived from the site's weekly ranking history: the number of consecutive weekly lists, ending with the latest, that show the current rank. (The site's own "consecutive weeks" figure describes the player's *best* rank, so it is not used.)
- **null** means the site lists no value. A player with no current rank (retired, or inactive for a long time) is reported as unranked.
- Responses are cached under `.cache/bwf_player` (ranking and profile for 24 h, the player index for 7 days), so re-running a cell makes no new requests. The site sits behind Cloudflare bot protection; the client rate-limits itself and stops at once if it is blocked.
- Known limitations and the full design are in `docs/PRD_master.md`.
- **Tournament history** (section 4): `won` is `1` or `0`, or empty for a bye (the player advanced without playing; the site counts it as a match won). A partner is empty in singles. The check line says whether the downloaded matches add up to the totals the site itself shows for each tournament.
- **Game details** (sections 4 and 5): for most events the site has the score after every rally and all statistics. For some it has only the game scores (small tournaments, and also some big or brand-new ones such as a team event in progress); those matches are stored with empty statistics, never zeros, and are marked `tracked = 0`. Byes and walkovers have no games and are not requested.
